In [0]:
# Databricks notebook source
# =============================================================================
# PRIMEINSURANCE — UC1: DQ INSIGHT ENGINE
# Reads : primeins.silver.dq_issues_log
# Writes: primeins.gold.dq_explanation_report_test  (test mode)
#         primeins.gold.dq_explanation_report        (production)
#
# PURPOSE:
#   Translates technical DQ logs into compliance-ready business explanations
#   using an LLM. Every row from dq_issues_log is sent to the model with a
#   structured prompt and the response is written back to Gold.
#
# LLM MODES (swap by changing TEST_MODE flag in Section 1):
#   TEST_MODE = True  → OpenRouter / arcee-ai/trinity-large-preview:free
#   TEST_MODE = False → Databricks Foundation Model (databricks-gpt-oss-20b)
#
# OUTPUT TABLE COLUMNS (minimum required by spec):
#   issue_id, table_name, rule_name, consequence, detail, suggested_fix,
#   logged_at, ai_explanation, generation_status, model_name, generated_at,
#   _gold_run_id
# =============================================================================

# COMMAND ----------

# ------------------------------------------------------------------
# 0. IMPORTS & SETUP
# ------------------------------------------------------------------
from openai import OpenAI
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from datetime import datetime
import json

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# ------------------------------------------------------------------
# TOGGLE THIS FLAG TO SWITCH LLM BACKEND
#   True  = OpenRouter (testing)
#   False = Databricks Foundation Model (production)
# ------------------------------------------------------------------
TEST_MODE = False

print(f"UC1 DQ Insight Engine  |  run_id    = {RUN_ID}")
print(f"                       |  test_mode = {TEST_MODE}")

# COMMAND ----------

# ------------------------------------------------------------------
# 1. LLM CLIENT SETUP
#
# TWO SETUPS — only the active one is used based on TEST_MODE.
# The other is kept commented so swapping is a one-line change.
# ------------------------------------------------------------------

if TEST_MODE:
    # ── TEST: OpenRouter (free tier, no Databricks billing) ──────────
    client = OpenAI(
        api_key  = "",
        base_url = "https://openrouter.ai/api/v1"
    )
    MODEL_ID = "arcee-ai/trinity-large-preview:free"
    TARGET_TABLE = "primeins.gold.dq_explanation_report_test"

else:
    #── PRODUCTION: Databricks Foundation Model ───────────────────────
    #Uncomment this block and set TEST_MODE = False to go live
    
    DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    WORKSPACE_URL    = spark.conf.get("spark.databricks.workspaceUrl")
    client = OpenAI(
        api_key  = DATABRICKS_TOKEN,
        base_url = f"https://{WORKSPACE_URL}/serving-endpoints"
    )
    MODEL_ID = "databricks-gpt-oss-20b"
    TARGET_TABLE = "primeins.gold.dq_explanation_report"

print(f"Model        : {MODEL_ID}")
print(f"Target table : {TARGET_TABLE}")

In [0]:
# COMMAND ----------

# ------------------------------------------------------------------
# 2. CONFIRM SOURCE TABLE EXISTS AND REVIEW SCHEMA
#
# dq_issues columns (actual schema):
#   issue_id         : String — UUID-style PK generated at Silver run time
#   table_name       : String — which Silver table was affected
#   column_name      : String — column involved ("N/A" if table-level)
#   issue_description: String — human-readable description of what was found
#   severity         : String — e.g. "Warning (Fix Applied)", "Error (Quarantined)"
#   affected_records : Integer — count of rows impacted
#
# NOTE: issue_id is already a natural PK (UUID) — no synthetic key needed.
# We retain row_number() as a numeric sequence alias for display/sorting only.
# ------------------------------------------------------------------
SOURCE_TABLE = "primeins.silver.dq_issues"

try:
    dq_raw = spark.table(SOURCE_TABLE)
    print(f"\nSource table : {SOURCE_TABLE}")
    print(f"Row count    : {dq_raw.count()}")
    print(f"Columns      : {dq_raw.columns}\n")
    dq_raw.show(10, truncate=False)
except Exception as e:
    raise RuntimeError(
        f"Source table {SOURCE_TABLE} not found. "
        f"Run the Silver pipeline first.\n{e}"
    )

# COMMAND ----------

# ------------------------------------------------------------------
# 3. ADD NUMERIC SEQUENCE FOR DISPLAY ORDERING
#
# issue_id is already a UUID — it serves as the natural PK.
# We add issue_seq (row_number) purely for readable ordering in the
# DQ dashboard. Sort by severity first (Errors before Warnings) so
# the most critical issues appear at the top of every report.
# ------------------------------------------------------------------
dq_issues = dq_raw.withColumn(
    "issue_seq",
    F.row_number().over(
        Window.orderBy(
            # Errors before Warnings — alphabetical on severity puts
            # "Error" before "Warning" naturally, which is what we want.
            F.col("severity"),
            F.col("table_name"),
            F.col("issue_id")          # tie-break on UUID for determinism
        )
    )
)

dq_rows = dq_issues.collect()
print(f"Issues to process: {len(dq_rows)}")

In [0]:
# COMMAND ----------

# ------------------------------------------------------------------
# 4. PROMPT DESIGN
#
# SYSTEM PROMPT — sets persona and enforces five-section structure.
# Five sections map directly to what compliance teams need:
#   FINDING        → what was found in plain English
#   BUSINESS IMPACT→ why it matters to the business
#   ROOT CAUSE     → what caused it
#   ACTION TAKEN   → what the pipeline did to fix it
#   PREVENTION     → what should be done to stop recurrence
#
# KEY DESIGN DECISIONS:
#   - Persona anchoring ("data governance advisor, not an engineer")
#     locks tone and vocabulary to compliance language
#   - Explicit numbered sections prevent the model from merging
#     all five points into a single unstructured paragraph
#   - "No preamble or sign-off" suppresses "Certainly! Here is..."
#   - consequence is pre-mapped to plain English before the prompt
#     so the model never sees "fix"/"warn"/"quarantine" raw codes
#   - Temperature 0.3 — low for consistent, auditable language
# ------------------------------------------------------------------

SYSTEM_PROMPT = """You are a data governance advisor writing for a compliance officer, not an engineer.

Your job is to explain data quality issues in plain business English.

Rules:
- Never use technical jargon (no "schema", "null", "regex", "pipeline", "DataFrame")
- Write in clear, direct sentences a non-technical reader can act on
- Use exactly these five section headers, in this order:

FINDING:
BUSINESS IMPACT:
ROOT CAUSE:
ACTION TAKEN:
PREVENTION:

Do not combine sections. Do not add a preamble or sign-off.
Respond only with the five sections."""

def build_prompt(row) -> str:
    """
    Builds the user prompt from a single dq_issues row.
    severity is mapped to plain English so the model never sees
    raw codes like "Error (Quarantined)".
    """
    severity_map = {
        "warning (fix applied)":   "The issue was automatically corrected by the system",
        "error (missing file)":    "A required data file was missing — no records were loaded",
        "error (quarantined)":     "Affected records were moved to a quarantine area for manual review",
    }
    severity_text = severity_map.get(
        str(row["severity"]).lower(),
        str(row["severity"])   # fallback: use value as-is if not in map
    )

    return f"""A data quality issue was detected in our insurance data system.
Explain it clearly for a compliance officer using the five sections.

SYSTEM AFFECTED    : {row['table_name']}
COLUMN AFFECTED    : {row['column_name']}
WHAT HAPPENED      : {row['issue_description']}
WHAT THE SYSTEM DID: {severity_text}
RECORDS AFFECTED   : {row['affected_records']}"""

In [0]:
# COMMAND ----------

# ------------------------------------------------------------------
# 5. RESPONSE PARSER
#
# databricks-gpt-oss-20b returns structured JSON blocks:
#   [{"type": "reasoning", "summary": [...]},
#    {"type": "text",      "text":    "actual answer"}]
#
# We extract only the "text" block. Returning the raw response would
# expose the model's reasoning chain.
#
# OpenRouter / arcee model returns a plain string — the parser handles
# both formats so swapping backends requires no code change here.
# ------------------------------------------------------------------
def parse_response(raw_content) -> str:
    """
    Extracts plain text from the model response.
    
    Databricks default model returns a plain string — returned as-is.
    If a future model returns JSON blocks (reasoning + text), the
    fallback handles it gracefully.
    """
    if isinstance(raw_content, str):
        return raw_content.strip()
    
    # Fallback for list-based block responses (non-default models only)
    if isinstance(raw_content, list):
        text_blocks = [
            b["text"] for b in raw_content
            if isinstance(b, dict) and b.get("type") == "text"
        ]
        if text_blocks:
            return "\n".join(text_blocks).strip()
    
    return str(raw_content).strip()


In [0]:
# COMMAND ----------

# ------------------------------------------------------------------
# 6. LLM CALL WRAPPER
#
# Each row is wrapped in try/except so one failed call does not abort
# the batch. Failed rows get status="error" and the error message in
# ai_explanation — the output table is always complete.
# ------------------------------------------------------------------
def call_llm(prompt: str) -> tuple[str, str]:
    try:
        response = client.chat.completions.create(
            model       = MODEL_ID,
            messages    = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": prompt},
            ],
            max_tokens  = 800,
            temperature = 0.3,
        )
        raw = response.choices[0].message.content
        return parse_response(raw), "success"
    except Exception as e:
        return f"LLM call failed: {str(e)}", "error"


# COMMAND ----------

# ------------------------------------------------------------------
# 7. PROCESS ALL ROWS
# ------------------------------------------------------------------
results = []
total   = len(dq_rows)

print(f"Processing {total} DQ issues...\n")

for i, row in enumerate(dq_rows, 1):
    print(f"  [{i}/{total}]  {row['table_name']} | {row['issue_description']}")

    prompt              = build_prompt(row)
    explanation, status = call_llm(prompt)

    results.append({
        # ── Original issue fields (preserved verbatim for traceability) ──
        "issue_id":           str(row["issue_id"]),          # UUID string — not int
        "issue_seq":          int(row["issue_seq"]),         # numeric display order
        "table_name":         str(row["table_name"]),
        "column_name":        str(row["column_name"]),
        "issue_description":  str(row["issue_description"]),
        "severity":           str(row["severity"]),
        "affected_records":   int(row["affected_records"]),
        # ── AI-generated fields ──────────────────────────────────────────
        "ai_explanation":     explanation,
        "generation_status":  status,
        "model_name":         MODEL_ID,
        "generated_at":       datetime.now().isoformat(),
        # ── Pipeline audit ───────────────────────────────────────────────
        "_gold_run_id":       RUN_ID,
    })

    print(f"         → {status}")

print(f"\nAll {total} issues processed.")



In [0]:
# ------------------------------------------------------------------
# 8. OUTPUT SCHEMA
#
# Explicit schema prevents Spark from inferring wrong types from Python
# dicts — especially important for long LLM text fields.
# ------------------------------------------------------------------
output_schema = T.StructType([
    T.StructField("issue_id",          T.StringType(),  nullable=False),  # UUID — not int
    T.StructField("issue_seq",         T.IntegerType(), nullable=True),   # display order
    T.StructField("table_name",        T.StringType(),  nullable=True),
    T.StructField("column_name",       T.StringType(),  nullable=True),
    T.StructField("issue_description", T.StringType(),  nullable=True),
    T.StructField("severity",          T.StringType(),  nullable=True),
    T.StructField("affected_records",  T.IntegerType(), nullable=True),
    T.StructField("ai_explanation",    T.StringType(),  nullable=True),
    T.StructField("generation_status", T.StringType(),  nullable=True),
    T.StructField("model_name",        T.StringType(),  nullable=True),
    T.StructField("generated_at",      T.StringType(),  nullable=True),
    T.StructField("_gold_run_id",      T.StringType(),  nullable=True),
])

In [0]:
# ------------------------------------------------------------------
# 9. WRITE TO GOLD
#
# Full overwrite — idempotent, safe to re-run.
# TARGET_TABLE switches between test and production based on TEST_MODE.
# ------------------------------------------------------------------
output_df = spark.createDataFrame(results, schema=output_schema)

(output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE))

final_count = spark.table(TARGET_TABLE).count()
print(f"\n✅  {TARGET_TABLE} — {final_count} rows written")

In [0]:
print("=" * 70)
print(f"SAMPLE OUTPUT — {TARGET_TABLE}")
print("=" * 70)

sample = spark.table(TARGET_TABLE).limit(3).collect()

for row in sample:
    print(f"\n{'─' * 70}")
    print(f"Issue ID        : {row['issue_id']}")
    print(f"Seq             : {row['issue_seq']}")
    print(f"Table           : {row['table_name']}")
    print(f"Column          : {row['column_name']}")
    print(f"Issue           : {row['issue_description']}")
    print(f"Severity        : {row['severity']}")
    print(f"Records Hit     : {row['affected_records']}")
    print(f"Generation Status: {row['generation_status']}")
    print(f"Model           : {row['model_name']}")
    print(f"Generated At    : {row['generated_at']}")
    print(f"\n{'─' * 35} AI EXPLANATION {'─' * 20}\n")
    print(row['ai_explanation'])

print(f"\n{'─' * 70}")
print(f"Run ID     : {RUN_ID}")
print(f"Total rows : {final_count}")

# ------------------------------------------------------------------
# 11. SCHEMA CONFIRMATION (for submission screenshot)
# ------------------------------------------------------------------
print("\nFinal table schema:")
spark.table(TARGET_TABLE).printSchema()